In [ ]:
from s3torchconnector import S3Checkpoint
import json

checkpoint = S3Checkpoint(region="us-west-2")
bucket_path = "<BUCKET-NAME>/ads/finalRanker/ray_train_run-2026-03-14_06-36-25"
checkpoint_snapshot_path = "checkpoint_manager_snapshot.json"
with checkpoint.reader(f"s3://{bucket_path}/{checkpoint_snapshot_path}") as reader:
    checkpoint_snapshot = json.loads(reader.read().decode("utf-8"))
checkpoint_snapshot

{'ray_version': '2.54.0',
 'checkpoint_results': [{'version': 0,
   'checkpoint_dir_name': 'checkpoint_2026-03-14_06-51-20.931149',
   'metrics': {'loss': 0.7394453108978513, 'epoch': 0}},
  {'version': 0,
   'checkpoint_dir_name': 'checkpoint_2026-03-14_07-01-15.377285',
   'metrics': {'loss': 0.7254336377941172, 'epoch': 1}},
  {'version': 0,
   'checkpoint_dir_name': 'checkpoint_2026-03-14_07-16-38.416655',
   'metrics': {'loss': 0.7212815541883965, 'epoch': 2}},
  {'version': 0,
   'checkpoint_dir_name': 'checkpoint_2026-03-14_07-26-04.759343',
   'metrics': {'loss': 0.7186235784073441, 'epoch': 3}},
  {'version': 0,
   'checkpoint_dir_name': 'checkpoint_2026-03-14_07-35-30.526058',
   'metrics': {'loss': 0.7166809521959994, 'epoch': 4}},
  {'version': 0,
   'checkpoint_dir_name': 'checkpoint_2026-03-14_07-44-54.340733',
   'metrics': {'loss': 0.7151954025712057, 'epoch': 5}},
  {'version': 0,
   'checkpoint_dir_name': 'checkpoint_2026-03-14_07-54-27.963325',
   'metrics': {'loss':

In [2]:
checkpoint_snapshot["latest_checkpoint_result"]["checkpoint_dir_name"]

'checkpoint_2026-03-14_08-21-45.762228'

In [1]:
from Ads import FinalRankerMMoE, FullRanker, EmbeddingLayer, DLRMTower

# eval metrics
# for "simplicity", Stack DLRMTower with the final ranker. This could potentially be used to share cached values across early-stage and final rankers.
# e.g. user and ad towers are shared between the rankers, but the final ranker also includes additional context-based (streaming features) tower.

bottom_mlp_layers = [512, 256, 64]
projection_layer = 128
emb_layers = {
    "uid": int(2e6),
    "campaign": 401, # +1 for null at index 400
    "cat1": 10,
    "cat2": 70,
    "cat3": 1829,
    "cat4": 21,
    "cat5": 51,
    "cat6": 30,
    "cat7": 57196,
    "cat8": 11,
    "cat9": 30,
}

sparse_features = { 
    "user" : ["uid", "last_n_click_campaigns_1D", "last_n_conversion_campaigns_1D"],
    "ad"   : ["campaign", "cat1", "cat2", "cat3", "cat4", "cat5", "cat6", "cat7", "cat8", "cat9"]
}


E = EmbeddingLayer(emb_layers=emb_layers, emb_dim=bottom_mlp_layers[-1])

# define user and ad tower params
base_params = {
    "bottom_mlp_layers" : bottom_mlp_layers,
    "projection_layer" : projection_layer,
    "embs" : E.embs,
    "dense_num" : 0
}
# copy to get independent dicts
u_params  = {**base_params, "sparse_num": len(sparse_features["user"])}
ad_params = {**base_params, "sparse_num": len(sparse_features["ad"])}

uTower = DLRMTower(**u_params)
adTower = DLRMTower(**ad_params)


expert_num = 4
expert_dims = [256]
# CTR, CVR
final_task_dims = [[256, 128, 64, 1], [256, 128, 64, 1]]
top_k = 2 # sparse MoE

final_ranker = FinalRankerMMoE(input_size=projection_layer*2, # input coming from user and ad towers
                               expert_num=expert_num,
                               expert_dims=expert_dims,
                               task_dims=final_task_dims,
                               top_k=top_k)

full_model = FullRanker(
    towers={"user": uTower, "ad": adTower},
    final_ranker=final_ranker
)


/Users/paataugrekhelidze/Projects/Recsys/infrastructure/distributed_compute/Ads/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-15 09:37:31,873	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-15 09:37:31,970	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [4]:
import torch
device = "cpu"
with checkpoint.reader(f"s3://{bucket_path}/{checkpoint_snapshot["latest_checkpoint_result"]["checkpoint_dir_name"]}/model.pt") as reader:
    state_dict = torch.load(reader,  map_location=device)

full_model.load_state_dict(state_dict["model_state_dict"])

<All keys matched successfully>

In [24]:
full_model

FullRanker(
  (towers): ModuleDict(
    (user): DLRMTower(
      (embs): ModuleDict(
        (uid): QREmbeddingBag([1415, 1414], [64, 64], mode=mean)
        (campaign): QREmbeddingBag([21, 20], [64, 64], mode=mean)
        (cat1): QREmbeddingBag([4, 3], [64, 64], mode=mean)
        (cat2): QREmbeddingBag([9, 8], [64, 64], mode=mean)
        (cat3): QREmbeddingBag([44, 42], [64, 64], mode=mean)
        (cat4): QREmbeddingBag([6, 4], [64, 64], mode=mean)
        (cat5): QREmbeddingBag([8, 7], [64, 64], mode=mean)
        (cat6): QREmbeddingBag([6, 5], [64, 64], mode=mean)
        (cat7): QREmbeddingBag([240, 239], [64, 64], mode=mean)
        (cat8): QREmbeddingBag([4, 3], [64, 64], mode=mean)
        (cat9): QREmbeddingBag([6, 5], [64, 64], mode=mean)
      )
      (projection): Linear(in_features=3, out_features=128, bias=True)
    )
    (ad): DLRMTower(
      (embs): ModuleDict(
        (uid): QREmbeddingBag([1415, 1414], [64, 64], mode=mean)
        (campaign): QREmbeddingBag([21, 2

In [6]:
from Ads import MultiplexedFinalRankerMMoE

multiplexed_model = MultiplexedFinalRankerMMoE(
                        input_size=projection_layer*2, # input coming from user and ad towers
                        expert_num=expert_num,
                        expert_dims=expert_dims,
                        task_dims=final_task_dims,
                        top_k=top_k
                    )

In [7]:
split_state = multiplexed_model.split_state_dict(model_state_dict=state_dict["model_state_dict"])
list(split_state)

['towers', 'final_ranker_shared', 'experts', 'other']

In [ ]:
from s3torchconnector import S3Checkpoint
import torch

# save model components separately
# required for multiplex serving

checkpoint = S3Checkpoint(region="us-west-2")
base_uri = "s3://<BUCKET-NAME>/ads/finalRanker/split/"

with checkpoint.writer(base_uri + "user_tower.pt") as writer:
    torch.save(split_state["towers"]["user"], writer)

with checkpoint.writer(base_uri + "ad_tower.pt") as writer:
    torch.save(split_state["towers"]["ad"], writer)

with checkpoint.writer(base_uri + "final_ranker_shared.pt") as writer:
    torch.save(split_state["final_ranker_shared"], writer)

for expert_id, expert_sd in split_state["experts"].items():
    with checkpoint.writer(base_uri + f"expert_{expert_id}.pt") as writer:
        torch.save(expert_sd, writer)

In [6]:
import torch

# load components separately, load towers
bottom_mlp_layers = [512, 256, 64]
projection_layer = 128
emb_layers = {
    "uid": int(2e6),
    "campaign": 401, # +1 for null at index 400
    "cat1": 10,
    "cat2": 70,
    "cat3": 1829,
    "cat4": 21,
    "cat5": 51,
    "cat6": 30,
    "cat7": 57196,
    "cat8": 11,
    "cat9": 30,
}

sparse_features = { 
    "user" : ["uid", "last_n_click_campaigns_1D", "last_n_conversion_campaigns_1D"],
    "ad"   : ["campaign", "cat1", "cat2", "cat3", "cat4", "cat5", "cat6", "cat7", "cat8", "cat9"]
}


E = EmbeddingLayer(emb_layers=emb_layers, emb_dim=bottom_mlp_layers[-1])

# define user and ad tower params
base_params = {
    "bottom_mlp_layers" : bottom_mlp_layers,
    "projection_layer" : projection_layer,
    "embs" : E.embs,
    "dense_num" : 0
}
# copy to get independent dicts
u_params  = {**base_params, "sparse_num": len(sparse_features["user"])}
ad_params = {**base_params, "sparse_num": len(sparse_features["ad"])}

uTower = DLRMTower(**u_params)
adTower = DLRMTower(**ad_params)


In [ ]:
device = "cpu"
bucket_path = "<BUCKET-NAME>/ads/finalRanker/split"
checkpoint = S3Checkpoint(region="us-west-2")

with checkpoint.reader(f"s3://{bucket_path}/user_tower.pt") as reader:
    user_state_dict = torch.load(reader,  map_location=device)
with checkpoint.reader(f"s3://{bucket_path}/ad_tower.pt") as reader:
    ad_state_dict = torch.load(reader,  map_location=device)


uTower.load_state_dict(user_state_dict)
adTower.load_state_dict(ad_state_dict)
uTower.eval()
adTower.eval()
towers = {"user": uTower, "ad": adTower}

In [2]:
# test dataloader
from datasets import Dataset
import os
from torch.utils.data import DataLoader
from Ads import AdsDataset
from functools import partial

filepath = "../../../datasets/criteo_attribution_dataset/processed"

train_df = Dataset.from_parquet(os.path.join(filepath, 'val/val.parquet'))

B = 8
WORKERS = 0

# Define voc size for user and campaign ids
user_v = emb_layers["uid"]
campaign_v = emb_layers["campaign"]


# Create a partial function with the arguments bound
collate_with_args = partial(AdsDataset._local_collate_batch, user_v=user_v, campaign_v=campaign_v)


train_dataset = AdsDataset(data= train_df)
train_ld = DataLoader(dataset=train_dataset, batch_size=B, shuffle=False, num_workers=WORKERS, collate_fn=collate_with_args)
total = len(train_df) // B

In [3]:
iterator = iter(train_ld)
sample = next(iterator)

In [ ]:
from Ads import Solver

with torch.no_grad():
    x, _ = Solver._build_x(sample)
    tower_outs = []
    for name, tower in towers.items():
        tower_outs.append(tower(**x[name]))

    # (B, P_u + P_ad + ...)
    combined = torch.cat(tower_outs, dim=1)
    routing = multiplexed_model.plan_routing(combined)
routing


In [37]:
len(routing["gate_weights"])

2

In [47]:
multiplexed_model.prepare_expert_inputs(combined, routing_plan=routing)

{0: {'inputs': tensor([[ 0.1670,  1.1559, -0.1822,  ..., -4.0597, -6.0420, -0.2087],
          [ 0.3782,  2.3746, -0.8197,  ..., -3.6900, -5.9674, -2.3348],
          [ 0.0660,  1.1604, -0.1477,  ..., -2.2848, -6.0177, -3.0775],
          ...,
          [-0.0693,  1.2890, -0.1773,  ..., -4.8328, -5.1865, -2.0992],
          [ 0.3293,  2.4142, -0.8274,  ..., -2.2773, -5.7224, -3.4848],
          [ 0.4031,  2.3545, -0.8158,  ..., -2.1486, -6.5584, -4.0333]]),
  'sample_indices': tensor([0, 1, 2, 3, 4, 5, 6, 7])},
 1: {'inputs': tensor([[ 0.3782,  2.3746, -0.8197,  ..., -3.6900, -5.9674, -2.3348],
          [ 0.0660,  1.1604, -0.1477,  ..., -2.2848, -6.0177, -3.0775],
          [ 0.2207,  2.5020, -0.8446,  ..., -2.9802, -4.6000, -2.0227],
          ...,
          [-0.0693,  1.2890, -0.1773,  ..., -4.8328, -5.1865, -2.0992],
          [ 0.3293,  2.4142, -0.8274,  ..., -2.2773, -5.7224, -3.4848],
          [ 0.4031,  2.3545, -0.8158,  ..., -2.1486, -6.5584, -4.0333]]),
  'sample_indices': t

In [8]:
import requests

payload = {
    "user": {
        "dense": None,
        "emb_indices": {
            "uid": [12345],
            "last_n_click_campaigns_1D": [17, 24, 31],
            "last_n_conversion_campaigns_1D": [24],
        },
        "emb_offsets": {
            "last_n_click_campaigns_1D_offset": [0],
            "last_n_conversion_campaigns_1D_offset": [0],
        },
    },
    "ad": {
        "dense": None,
        "emb_indices": {
            "campaign": [24],
            "cat1": [3],
            "cat2": [12],
            "cat3": [145],
            "cat4": [7],
            "cat5": [9],
            "cat6": [4],
            "cat7": [1024],
            "cat8": [2],
            "cat9": [6],
        },
        "emb_offsets": {},
    },
}

response = requests.post(
    "http://127.0.0.1:8000/",
    json=payload,
    timeout=30,
)

response.raise_for_status()
print(response.json())

{'logits': [[1.3533568382263184], [-0.3692680895328522]], 'topk_indices': [[[3, 0]], [[0, 2]]], 'topk_weights': [[[0.9989914298057556, 0.0010085784597322345]], [[0.9999459981918335, 5.4009538871468976e-05]]]}


In [9]:
Solver._serve_build_x(sample)[0]

{'user': {'dense': None,
  'emb_indices': {'uid': [1874503,
    859866,
    1784603,
    1254421,
    1524937,
    272050,
    451963,
    1333272],
   'last_n_click_campaigns_1D': [393, 401, 26, 401, 401, 102, 401, 401],
   'last_n_conversion_campaigns_1D': [401, 401, 401, 401, 401, 401, 401, 401]},
  'emb_offsets': {'last_n_click_campaigns_1D_offset': [0, 1, 2, 3, 4, 5, 6, 7],
   'last_n_conversion_campaigns_1D_offset': [0, 1, 2, 3, 4, 5, 6, 7]}},
 'ad': {'dense': None,
  'emb_indices': {'campaign': [242, 399, 132, 85, 26, 399, 3, 188],
   'cat1': [5, 0, 8, 4, 8, 8, 1, 8],
   'cat2': [60, 23, 46, 54, 23, 23, 23, 23],
   'cat3': [37, 670, 897, 1826, 622, 670, 1828, 97],
   'cat4': [17, 17, 17, 17, 17, 17, 17, 17],
   'cat5': [39, 7, 47, 48, 47, 7, 9, 38],
   'cat6': [29, 1, 26, 1, 1, 1, 1, 1],
   'cat7': [19304, 19311, 26457, 16276, 29518, 44648, 32374, 53113],
   'cat8': [8, 9, 8, 10, 5, 4, 5, 2],
   'cat9': [18, 25, 17, 25, 25, 19, 25, 25]},
  'emb_offsets': {}}}

In [37]:
from Ads import Solver

# if forwarded to local port 8000
response = requests.post(
    "http://127.0.0.1:8000/",
    json=Solver._serve_build_x(sample)[0],
    timeout=30,
)

response.raise_for_status()
print(response.json())


{'logits': [[-0.8107800483703613, 0.008037686347961426, -1.1683635711669922, -1.6171469688415527, -0.6703653931617737, 0.019025057554244995, -0.9120479226112366, -0.6994547843933105], [-4.117852210998535, -4.130635738372803, -6.796058177947998, -5.670886516571045, -4.778652191162109, -6.231645584106445, -6.212043285369873, -5.749763011932373]], 'topk_indices': [[[3, 0], [3, 0], [3, 0], [3, 0], [3, 0], [3, 0], [3, 0], [3, 0]], [[0, 2], [0, 2], [0, 2], [0, 2], [0, 2], [0, 2], [0, 1], [0, 3]]], 'topk_weights': [[[0.9996906518936157, 0.0003093592240475118], [0.9999847412109375, 1.523626360722119e-05], [0.9977232217788696, 0.002276769606396556], [0.9999916553497314, 8.33693138702074e-06], [0.9999750256538391, 2.5003331757034175e-05], [0.9995772838592529, 0.0004227218159940094], [0.9999945163726807, 5.489713657880202e-06], [0.9999896287918091, 1.0360606211179402e-05]], [[0.9999927878379822, 7.217305665108142e-06], [0.9999971985816956, 2.821434918587329e-06], [0.9999980926513672, 1.9127521682